In [ ]:
import os
import sqlite3
import pandas as pd

DB_PATH = "/content/NO_SQL.db"
OUT_DIR = "neo4j_csv"
os.makedirs(OUT_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

# --- Tables dimensionnelles ---
dept = pd.read_sql_query("SELECT Code_dept AS code FROM Département", conn)
dept.to_csv(f"{OUT_DIR}/departements.csv", index=False)

service = pd.read_sql_query("""
SELECT code_service AS id, nom_service AS nom
FROM Type_service
""", conn)
service.to_csv(f"{OUT_DIR}/services.csv", index=False)

perim = pd.read_sql_query("""
SELECT Code_perimetre AS id, Nom_perimetre AS nom
FROM perimetre
""", conn)
perim.to_csv(f"{OUT_DIR}/perimetres.csv", index=False)

brig = pd.read_sql_query("""
SELECT
  Code_brigade AS id,
  Nom_brigade AS nom,
  Code_dept AS dep_code,
  id_service AS service_id,
  Code_perimetre AS perimetre_id
FROM base_brigade
""", conn)

brig.to_csv(f"{OUT_DIR}/brigades.csv", index=False)

infrac = pd.read_sql_query("""
SELECT
  Code_infrac AS id,
  Nom_Infrac AS nom
FROM Infraction
""", conn)
infrac.to_csv(f"{OUT_DIR}/infractions.csv", index=False)

annees = pd.read_sql_query("""
SELECT DISTINCT annee AS valeur FROM Enregistrement
ORDER BY valeur
""", conn)
annees.to_csv(f"{OUT_DIR}/annees.csv", index=False)

# --- Enregistrements -> Faits ---
faits = pd.read_sql_query("""
SELECT
  e.annee,
  e.Code_dept,
  e.Nom_service,
  e.Nom_perimetre,
  e.Nom_brigade,
  e.Code_infrac,
  e.nombre_fait
FROM Enregistrement e
""", conn)

# Table mapping Nom_perimetre -> Code_perimetre
per_map = pd.read_sql_query("SELECT Code_perimetre, Nom_perimetre FROM perimetre", conn)
svc_map = pd.read_sql_query("SELECT code_service, nom_service FROM Type_service", conn)
brig_map = pd.read_sql_query("""
SELECT Code_brigade, Nom_brigade, Code_dept, id_service, Code_perimetre
FROM base_brigade
""", conn)

# join service
faits = faits.merge(svc_map, left_on="Nom_service", right_on="nom_service", how="left")
faits.rename(columns={"code_service": "service_id"}, inplace=True)

# join perimetre name -> id
faits = faits.merge(per_map, left_on="Nom_perimetre", right_on="Nom_perimetre", how="left")
faits.rename(columns={"Code_perimetre": "perimetre_id"}, inplace=True)

# join brigade -> Code_brigade
faits = faits.merge(
    brig_map,
    left_on=["Nom_brigade", "Code_dept", "service_id", "perimetre_id"],
    right_on=["Nom_brigade", "Code_dept", "id_service", "Code_perimetre"],
    how="left"
)

# check mapping
missing = faits["Code_brigade"].isna().sum()
print("Brigades non mappées:", missing)

# créer un id de fait
faits["fait_id"] = (
    faits["annee"].astype(str) + "|" +
    faits["Code_dept"].astype(str) + "|" +
    faits["service_id"].astype(str) + "|" +
    faits["perimetre_id"].fillna(-1).astype(int).astype(str) + "|" +
    faits["Code_brigade"].fillna(-1).astype(int).astype(str) + "|" +
    faits["Code_infrac"].astype(str)
)

faits_out = faits[[
    "fait_id",
    "annee",
    "Code_dept",
    "service_id",
    "perimetre_id",
    "Code_brigade",
    "Code_infrac",
    "nombre_fait"
]].rename(columns={
    "Code_dept": "dep_code",
    "Code_brigade": "brigade_id",
    "Code_infrac": "infraction_id",
    "nombre_fait": "nb"
})

faits_out.to_csv(f"{OUT_DIR}/faits.csv", index=False)

# --- Population ---
pop = pd.read_sql_query("""
SELECT
  `Code département` AS dep_code,
  `Année` AS annee,
  `Population totale` AS total,
  `Total hommes` AS hommes,
  `Total femmes` AS femmes
FROM Population_dept
""", conn)

# id population
pop["pop_id"] = pop["dep_code"].astype(str) + "|" + pop["annee"].astype(str)
pop.to_csv(f"{OUT_DIR}/population.csv", index=False)

print(" CSV générés dans:", OUT_DIR)

Brigades non mappées: 3132
✅ CSV générés dans: neo4j_csv


OK, j’ai ouvert ton fichier **SQLite** `/mnt/data/NO_SQL.db`. Il contient déjà un modèle relationnel propre avec ces tables :

* `Département(Code_dept)`
* `Type_service(code_service, nom_service)`  *(PN / GN)*
* `perimetre(Code_perimetre, Nom_perimetre)`
* `base_brigade(Code_brigade, Nom_brigade, Code_perimetre, Code_dept, id_service)`
* `Infraction(Code_infrac, Nom_Infrac)`
* `Enregistrement(annee, Nom_service, Code_dept, Nom_perimetre, Nom_brigade, Code_infrac, nom_infrac, nombre_fait)`
* `Population_dept(Code département, Année, Population totale, Total hommes, Total femmes)`

🎯 **Objectif graphe (Neo4j)** (adaptation directe) :

### Nœuds

* `(:Departement {code})`
* `(:Service {id, nom})`
* `(:Perimetre {id, nom})` *(surtout PN)*
* `(:Brigade {id, nom})` *(= CSP / base_brigade)*
* `(:Infraction {id, nom})`
* `(:Annee {valeur})`
* `(:Fait {id, nb})` *(un enregistrement agrégé)*
* `(:Population {id, total, hommes, femmes})` *(optionnel mais utile)*

### Relations

* `(s:Service)-[:COUVRE]->(d:Departement)`
* `(b:Brigade)-[:APPARTIENT_A]->(d)`
* `(b)-[:DEPEND_DE]->(s)`
* `(b)-[:DANS_PERIMETRE]->(p:Perimetre)` *(si présent)*
* `(b)-[:ENREGISTRE]->(f:Fait)`
* `(f)-[:SE_DEROULE_DANS]->(d)`
* `(f)-[:CONCERNE]->(i:Infraction)`
* `(f)-[:OBSERVE_EN]->(a:Annee)`
* `(pop:Population)-[:POP_DE]->(d)` et `(pop)-[:POP_ANNEE]->(a)`

---

# 1) Script Python : export SQLite ➜ CSV “Neo4j-ready”

Ce script lit ton `.db` et génère des CSV prêts pour Neo4j (dossier `import/`).

👉 À exécuter sur ta machine (ou Colab) :

```python
import os
import sqlite3
import pandas as pd

DB_PATH = "NO_SQL.db"          # mets le bon chemin
OUT_DIR = "neo4j_csv"          # tu copieras ensuite ces CSV dans le dossier import Neo4j
os.makedirs(OUT_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

# --- Tables dimensionnelles ---
dept = pd.read_sql_query("SELECT Code_dept AS code FROM Département", conn)
dept.to_csv(f"{OUT_DIR}/departements.csv", index=False)

service = pd.read_sql_query("""
SELECT code_service AS id, nom_service AS nom
FROM Type_service
""", conn)
service.to_csv(f"{OUT_DIR}/services.csv", index=False)

perim = pd.read_sql_query("""
SELECT Code_perimetre AS id, Nom_perimetre AS nom
FROM perimetre
""", conn)
perim.to_csv(f"{OUT_DIR}/perimetres.csv", index=False)

brig = pd.read_sql_query("""
SELECT
  Code_brigade AS id,
  Nom_brigade AS nom,
  Code_dept AS dep_code,
  id_service AS service_id,
  Code_perimetre AS perimetre_id
FROM base_brigade
""", conn)
# perimetre_id peut être NULL -> garder vide
brig.to_csv(f"{OUT_DIR}/brigades.csv", index=False)

infrac = pd.read_sql_query("""
SELECT
  Code_infrac AS id,
  Nom_Infrac AS nom
FROM Infraction
""", conn)
infrac.to_csv(f"{OUT_DIR}/infractions.csv", index=False)

annees = pd.read_sql_query("""
SELECT DISTINCT annee AS valeur FROM Enregistrement
ORDER BY valeur
""", conn)
annees.to_csv(f"{OUT_DIR}/annees.csv", index=False)

# --- Enregistrements -> Faits ---
# On mappe Enregistrement -> Code_brigade via base_brigade (+ perimetre + service)
faits = pd.read_sql_query("""
SELECT
  e.annee,
  e.Code_dept,
  e.Nom_service,
  e.Nom_perimetre,
  e.Nom_brigade,
  e.Code_infrac,
  e.nombre_fait
FROM Enregistrement e
""", conn)

# Table mapping Nom_perimetre -> Code_perimetre
per_map = pd.read_sql_query("SELECT Code_perimetre, Nom_perimetre FROM perimetre", conn)
svc_map = pd.read_sql_query("SELECT code_service, nom_service FROM Type_service", conn)
brig_map = pd.read_sql_query("""
SELECT Code_brigade, Nom_brigade, Code_dept, id_service, Code_perimetre
FROM base_brigade
""", conn)

# join service
faits = faits.merge(svc_map, left_on="Nom_service", right_on="nom_service", how="left")
faits.rename(columns={"code_service": "service_id"}, inplace=True)

# join perimetre name -> id (optionnel)
faits = faits.merge(per_map, left_on="Nom_perimetre", right_on="Nom_perimetre", how="left")
faits.rename(columns={"Code_perimetre": "perimetre_id"}, inplace=True)

# join brigade -> Code_brigade (clé unique: Nom_brigade + dept + service_id + perimetre_id)
faits = faits.merge(
    brig_map,
    left_on=["Nom_brigade", "Code_dept", "service_id", "perimetre_id"],
    right_on=["Nom_brigade", "Code_dept", "id_service", "Code_perimetre"],
    how="left"
)

# check mapping
missing = faits["Code_brigade"].isna().sum()
print("Brigades non mappées:", missing)

# créer un id de fait (stable)
# (si tu ré-exportes, tu gardes le même id grâce à la concat)
faits["fait_id"] = (
    faits["annee"].astype(str) + "|" +
    faits["Code_dept"].astype(str) + "|" +
    faits["service_id"].astype(str) + "|" +
    faits["perimetre_id"].fillna(-1).astype(int).astype(str) + "|" +
    faits["Code_brigade"].fillna(-1).astype(int).astype(str) + "|" +
    faits["Code_infrac"].astype(str)
)

faits_out = faits[[
    "fait_id",
    "annee",
    "Code_dept",
    "service_id",
    "perimetre_id",
    "Code_brigade",
    "Code_infrac",
    "nombre_fait"
]].rename(columns={
    "Code_dept": "dep_code",
    "Code_brigade": "brigade_id",
    "Code_infrac": "infraction_id",
    "nombre_fait": "nb"
})

faits_out.to_csv(f"{OUT_DIR}/faits.csv", index=False)

# --- Population ---
pop = pd.read_sql_query("""
SELECT
  `Code département` AS dep_code,
  `Année` AS annee,
  `Population totale` AS total,
  `Total hommes` AS hommes,
  `Total femmes` AS femmes
FROM Population_dept
""", conn)

# id population
pop["pop_id"] = pop["dep_code"].astype(str) + "|" + pop["annee"].astype(str)
pop.to_csv(f"{OUT_DIR}/population.csv", index=False)

print("✅ CSV générés dans:", OUT_DIR)
```

➡️ Ensuite tu copies les fichiers CSV (`neo4j_csv/*.csv`) dans **le dossier import** de Neo4j Desktop.

---

# 2) Cypher Neo4j : création du graphe depuis les CSV

> (sans APOC, compatible Neo4j 5)

### 2.1 Contraintes

```cypher
CREATE CONSTRAINT IF NOT EXISTS FOR (d:Departement) REQUIRE d.code IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (s:Service) REQUIRE s.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (p:Perimetre) REQUIRE p.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (b:Brigade) REQUIRE b.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (i:Infraction) REQUIRE i.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (a:Annee) REQUIRE a.valeur IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (f:Fait) REQUIRE f.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (pop:Population) REQUIRE pop.id IS UNIQUE;
```

### 2.2 Import des nœuds

```cypher
LOAD CSV WITH HEADERS FROM 'file:///departements.csv' AS r
MERGE (:Departement {code: r.code});

LOAD CSV WITH HEADERS FROM 'file:///services.csv' AS r
MERGE (:Service {id: toInteger(r.id)}) SET _.nom = r.nom;

LOAD CSV WITH HEADERS FROM 'file:///perimetres.csv' AS r
MERGE (:Perimetre {id: toInteger(r.id)}) SET _.nom = r.nom;

LOAD CSV WITH HEADERS FROM 'file:///brigades.csv' AS r
MERGE (b:Brigade {id: toInteger(r.id)})
SET b.nom = r.nom;

LOAD CSV WITH HEADERS FROM 'file:///infractions.csv' AS r
MERGE (i:Infraction {id: toInteger(r.id)})
SET i.nom = r.nom;

LOAD CSV WITH HEADERS FROM 'file:///annees.csv' AS r
MERGE (:Annee {valeur: toInteger(r.valeur)});

LOAD CSV WITH HEADERS FROM 'file:///population.csv' AS r
MERGE (pop:Population {id: r.pop_id})
SET pop.total = toInteger(r.total),
    pop.hommes = toInteger(r.hommes),
    pop.femmes = toInteger(r.femmes);
```

### 2.3 Relations structurelles (brigade/service/dept/perimetre)

```cypher
LOAD CSV WITH HEADERS FROM 'file:///brigades.csv' AS r
MATCH (b:Brigade {id: toInteger(r.id)})
MATCH (d:Departement {code: r.dep_code})
MATCH (s:Service {id: toInteger(r.service_id)})
MERGE (b)-[:APPARTIENT_A]->(d)
MERGE (b)-[:DEPEND_DE]->(s)
MERGE (s)-[:COUVRE]->(d);

LOAD CSV WITH HEADERS FROM 'file:///brigades.csv' AS r
WITH r
WHERE r.perimetre_id IS NOT NULL AND trim(r.perimetre_id) <> ''
MATCH (b:Brigade {id: toInteger(r.id)})
MATCH (p:Perimetre {id: toInteger(r.perimetre_id)})
MERGE (b)-[:DANS_PERIMETRE]->(p);
```

### 2.4 Import des faits + relations (à faire par année pour éviter la mémoire)

Comme ta table `Enregistrement` est grosse (~588k lignes), fais par année :

```cypher
:param annee => 2012
```

**Faits pour une année**

```cypher
LOAD CSV WITH HEADERS FROM 'file:///faits.csv' AS r
WITH r
WHERE toInteger(r.annee) = $annee

MERGE (f:Fait {id: r.fait_id})
SET f.nb = toInteger(r.nb)

WITH r, f
MATCH (a:Annee {valeur: toInteger(r.annee)})
MATCH (d:Departement {code: r.dep_code})
MATCH (i:Infraction {id: toInteger(r.infraction_id)})
MATCH (b:Brigade {id: toInteger(r.brigade_id)})
MERGE (b)-[:ENREGISTRE]->(f)
MERGE (f)-[:SE_DEROULE_DANS]->(d)
MERGE (f)-[:CONCERNE]->(i)
MERGE (f)-[:OBSERVE_EN]->(a);
```

**Lier périmètre aux faits si présent**

```cypher
LOAD CSV WITH HEADERS FROM 'file:///faits.csv' AS r
WITH r
WHERE toInteger(r.annee) = $annee
  AND r.perimetre_id IS NOT NULL AND trim(r.perimetre_id) <> ''

MATCH (f:Fait {id: r.fait_id})
MATCH (p:Perimetre {id: toInteger(r.perimetre_id)})
MERGE (f)-[:DANS_PERIMETRE]->(p);
```

Et tu répètes `:param annee => 2013` … jusqu’à 2022.

### 2.5 Relations population

```cypher
LOAD CSV WITH HEADERS FROM 'file:///population.csv' AS r
MATCH (pop:Population {id: r.pop_id})
MATCH (d:Departement {code: toString(r.dep_code)})
MATCH (a:Annee {valeur: toInteger(r.annee)})
MERGE (pop)-[:POP_DE]->(d)
MERGE (pop)-[:POP_ANNEE]->(a);
```

---

## Si tu veux, je te fais l’adaptation “pile à ton schéma”

Il me manque juste **une info** : dans votre schéma final, vous appelez le nœud “Infraction” ou “TypeCrime” ?
✅ Si tu me dis le label exact, je te renvoie le Cypher renommé à 100% (labels + types de relations identiques à votre MCD).
